In [ ]:
import numpy as np
from scipy.signal import resample_poly, spectrogram
import matplotlib.pyplot as plt
import sys
import os
import torch
# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa, MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

In [ ]:
sf = 9
bw = 125000
fs = 250000
lora_init = LoRa(sf, bw)
input_row = 512
input_col = 33
## HOW TO LOAD WEIGHT
layers = [input_col * input_row, 1024, 256] # <-- must match training

multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)
# for i, bam in enumerate(multi_bam.bams):
#     bam.W = np.load(f"weight_3840_1024_256_model2/weights_layer_{i}.npy")

for i, bam in enumerate(multi_bam.bams):
    w_np = np.load(f"weight_{input_row*input_col}_1024_256/weights_layer_{i}.npy")
    bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
    print(f"Loaded layer {i}: shape {bam.W.shape}")


In [ ]:
symbol1 = 20
symbol2 = 300
# EXAMPLE OF COMPRESS and DECOMPRESS
seed = 26
x1 = lora_init.gen_symbol_fs(symbol1, sf=sf, bw=bw, Fs=1000000)  # you had Fs=int(bw*8)=1e6

x2 = lora_init.gen_symbol_fs(symbol1, sf=sf, bw=bw, Fs=1000000)  # you had Fs=int(bw*8)=1e6

original_pisan = create_spectrogram_from_torch(x1,sf,bw,1000000,input_row,input_col,0,0,1,None)

original_pisan2 = create_spectrogram_from_torch(x2,sf,bw,1000000,input_row,input_col,0,0,1,None)
show_multiple_spectrograms([original_pisan,original_pisan2],["a","GT"],2)
snr = 12
x = lora_init.awgn_iq_with_seed(x1,snr,seed)
x22 = lora_init.awgn_iq_with_seed(x2,snr,seed)
x_ds,fs_new = downsampling(x,1000000,4)
# image_ori_noise,null,null = create_spectrogram_npy(x_down,fs_new_down,0,0,1,None)

# x2 = apply_cfo(x, Fs=fs_new_down, freq_offset_hz=300)
# x2 = apply_phase_noise(x2, Fs=fs_new_down, linewidth_hz=50)
# x2 = multipath_rayleigh(x2, [0, 5], 6)
# x2 = band_limited_noise(x2, Fs=fs_new_down, low_hz=110e3, high_hz=120e3, snr_db=20)
# x2 = quantize_iq(x2, nbits=8)
# x2 = time_varying_rayleigh(x2,10,fs_new_down)
# x2 = hard_clip(x2, max_amp=0.1)

#### STEP 1 PREPROCESS : DOWNSAMPLING ########
# x_ds,fs_new = downsampling(x2,fs,4)

#### STEP 2 Create Spectrogram ########
aa = create_spectrogram_from_torch(x_ds,sf,bw,250000,input_row,input_col,0,0,1,None)
bb = create_spectrogram_from_torch(x22,sf,bw,1000000,input_row,input_col,0,0,1,None)
#### STEP 3 FLATTEN INPUT ########
flat = aa.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
out1 = multi_bam.compress(flat)
out2 = multi_bam.decompress(out1)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat = out2.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec = reco_flat.reshape(input_row, input_col)

flatbb = bb.flatten().reshape(1, -1) # BEFORE COMPRESS MUST

out1bb = multi_bam.compress(flatbb)
out2bb = multi_bam.decompress(out1bb)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flatbb = out2bb.reshape(-1)  # AFTER DECOMPRESS MUST
reco_specbb = reco_flatbb.reshape(input_row, input_col)

array_a = [aa,reco_spec,original_pisan]

show_multiple_spectrograms(array_a,[f'SNR {snr} sym {symbol1}',f"decompress {input_row}x{input_col}","GT"],2)

show_multiple_spectrograms([reco_spec,reco_specbb],[f'SNR {snr} sym {symbol1}',f"decompress {input_row}x{input_col}","GT"],2)